### Realsense d405 카메라 OBB

In [8]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple
from dataclasses import dataclass
import time

import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration (Embedded for Test)
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.85
    iou_thres: float = 0.75
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # box real size (mm)
    box_w_mm: float = 230.0
    box_h_mm: float = 95.0

    size_rel_err_max: float = 0.25

    # sampling
    avg_n: int = 10
    timeout_sec: float = 25.0

    # depth ROI (meters)
    roi_margin_px: float = 6.0
    min_roi_pixels: int = 120
    mad_thres_m: float = 0.020
    depth_min_m: float = 0.15
    depth_max_m: float = 3.00

    # sanity filters
    z_range_mm: Tuple[float, float] = (150.0, 1200.0)
    
    jump_xy_mm: float = 35.0
    jump_z_mm: float = 60.0
    jump_ang_deg: float = 10.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    show_preview: bool = True
    preview_win_name: str = "OBB + center"
    show_overlay: bool = True
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2
    print_selected_each_accept: bool = True


# 기본 설정 인스턴스 생성
DEFAULT_VISION_CONFIG = VisionConfig(
    # 주의: 실제 모델 경로가 맞는지 확인하세요
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)


# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def poly_shrink_towards_center(poly4x2: np.ndarray, margin_px: float):
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    v = p - c
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-6
    return p - (v / norm) * margin_px


def depth_roi_stats(depth_u16: np.ndarray, depth_scale: float, poly4x2: np.ndarray, cfg: VisionConfig):
    h, w = depth_u16.shape[:2]
    poly = np.round(poly4x2).astype(np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.reshape(-1, 1, 2)], 255)

    d = depth_u16[mask == 255].astype(np.float32) * depth_scale
    d = d[(d > 0) & (d >= cfg.depth_min_m) & (d <= cfg.depth_max_m)]
    if d.size == 0:
        return 0.0, 0.0, 0

    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    return med, mad, int(d.size)


def edges_long_short_px(poly4x2: np.ndarray):
    p = poly4x2.astype(np.float32)
    edges = [np.linalg.norm(p[(i + 1) % 4] - p[i]) for i in range(4)]
    return float(max(edges)), float(min(edges))


def estimate_Z_from_size(poly4x2: np.ndarray, intr, W_mm: float, H_mm: float) -> float:
    long_px, short_px = edges_long_short_px(poly4x2)
    W_m = W_mm / 1000.0
    H_m = H_mm / 1000.0

    if W_m >= H_m:
        Z1 = (intr.fx * W_m) / max(long_px, 1e-6)
        Z2 = (intr.fy * H_m) / max(short_px, 1e-6)
    else:
        Z1 = (intr.fx * H_m) / max(long_px, 1e-6)
        Z2 = (intr.fy * W_m) / max(short_px, 1e-6)

    return float(0.5 * (Z1 + Z2))  # meters


def XY_from_pixel_and_Z(cx: int, cy: int, intr, Z_m: float):
    X = (cx - intr.ppx) / intr.fx * Z_m
    Y = (cy - intr.ppy) / intr.fy * Z_m
    return float(X), float(Y)  # meters


def size_consistency_check(poly4x2, intr, Z_use_m, W_mm, H_mm, rel_err_max=0.25):
    long_px, short_px = edges_long_short_px(poly4x2)

    L1_mm = (long_px * Z_use_m / intr.fx) * 1000.0
    L2_mm = (short_px * Z_use_m / intr.fy) * 1000.0

    W_big = max(W_mm, H_mm)
    H_sml = min(W_mm, H_mm)

    err1 = abs(L1_mm - W_big) / max(1e-6, W_big)
    err2 = abs(L2_mm - H_sml) / max(1e-6, H_sml)

    return (err1 <= rel_err_max) and (err2 <= rel_err_max)


def obb_angle_deg_upright0_rightplus(poly4x2: np.ndarray) -> float:
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    q = p - c
    cov = np.cov(q.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    v = eigvecs[:, np.argmax(eigvals)].astype(np.float32)

    vx, vy = float(v[0]), float(v[1])
    if vy < 0:
        vx, vy = -vx, -vy

    angle = float(np.degrees(np.arctan2(vx, vy)))
    angle = -angle
    return angle


def is_jump(prev, cur, cfg: VisionConfig):
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg:
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, angle, cfg: VisionConfig):
    if not cfg.show_overlay:
        return

    line1 = f"cam X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f}  (mm)"
    line2 = f"angle {angle:+.2f} deg"

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)


def measure_box(cfg: VisionConfig = DEFAULT_VISION_CONFIG) -> Optional[Dict[str, Any]]:
    model = YOLO(cfg.model_path)

    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = float(depth_sensor.get_depth_scale())

    temporal = rs.temporal_filter()
    spatial = rs.spatial_filter()
    hole = rs.hole_filling_filter()

    spatial.set_option(rs.option.filter_magnitude, 2)
    spatial.set_option(rs.option.filter_smooth_alpha, 0.5)
    spatial.set_option(rs.option.filter_smooth_delta, 20)

    accepted_xyza = []  # [[Xmm,Ymm,Zmm,angle], ...]
    prev_valid = None
    consec_skips = 0
    t0 = time.time()

    if cfg.show_preview:
        cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)

    last_disp = {"Xmm": None, "Ymm": None, "Zmm": None, "angle": None}

    try:
        while True:
            if time.time() - t0 > cfg.timeout_sec:
                print("Timeout reached.")
                return None

            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            depth_frame = spatial.process(depth_frame).as_depth_frame()
            depth_frame = temporal.process(depth_frame).as_depth_frame()
            depth_frame = hole.process(depth_frame).as_depth_frame()

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            depth_u16 = np.asanyarray(depth_frame.get_data())

            results = model.predict(frame, imgsz=cfg.imgsz, conf=cfg.conf_thres, iou=cfg.iou_thres, verbose=False)
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    clss  = obb.cls.cpu().numpy().astype(int)

                    for poly8, cf, ci in zip(polys, confs, clss):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, -float(cf), float(cf), int(ci), poly, cx_det, cy_det))

            if not candidates:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

                if cfg.show_preview:
                    vis = frame
                    if cfg.show_overlay and last_disp["Xmm"] is not None:
                        vis = vis.copy()
                        draw_overlay_xyz_angle(vis, last_disp["Xmm"], last_disp["Ymm"], last_disp["Zmm"], last_disp["angle"], cfg)
                    cv2.imshow(cfg.preview_win_name, vis)
                    if (cv2.waitKey(1) & 0xFF) == 27:
                        return None
                continue

            candidates.sort()
            dist2, _ncf, cf, ci, poly, cx_det_f, cy_det_f = candidates[0]
            chosen_dist_px = float(np.sqrt(dist2))
            num_boxes = len(candidates)

            cx = clamp(int(round(cx_det_f)), 0, cfg.width - 1)
            cy = clamp(int(round(cy_det_f)), 0, cfg.height - 1)

            vis = frame.copy()
            poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
            cv2.circle(vis, (cx, cy), 5, (0, 0, 255), -1)

            poly_shrunk = poly_shrink_towards_center(poly, cfg.roi_margin_px)
            poly_shrunk[:, 0] = np.clip(poly_shrunk[:, 0], 0, cfg.width - 1)
            poly_shrunk[:, 1] = np.clip(poly_shrunk[:, 1], 0, cfg.height - 1)

            Z_roi_m, mad_m, roi_n = depth_roi_stats(depth_u16, depth_scale, poly_shrunk, cfg)
            Z_size_m = estimate_Z_from_size(poly, intr, cfg.box_w_mm, cfg.box_h_mm)

            depth_ok = (Z_roi_m > 0.0 and roi_n >= cfg.min_roi_pixels and mad_m <= cfg.mad_thres_m)
            if depth_ok:
                alpha = clamp(0.85 - (mad_m / max(1e-6, cfg.mad_thres_m)) * 0.35, 0.55, 0.90)
                Z_use_m = alpha * Z_roi_m + (1.0 - alpha) * Z_size_m
            else:
                Z_use_m = Z_size_m

            Z_use_mm = Z_use_m * 1000.0
            if not (cfg.z_range_mm[0] <= Z_use_mm <= cfg.z_range_mm[1]):
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

                if cfg.show_preview:
                    if cfg.show_overlay and last_disp["Xmm"] is not None:
                        draw_overlay_xyz_angle(vis, last_disp["Xmm"], last_disp["Ymm"], last_disp["Zmm"], last_disp["angle"], cfg)
                    cv2.imshow(cfg.preview_win_name, vis)
                    if (cv2.waitKey(1) & 0xFF) == 27:
                        return None
                continue

            X_m, Y_m = XY_from_pixel_and_Z(cx, cy, intr, Z_use_m)
            angle = obb_angle_deg_upright0_rightplus(poly)

            cur = {
                "Xmm": X_m * 1000.0,
                "Ymm": Y_m * 1000.0,
                "Zmm": Z_use_m * 1000.0,
                "angle": float(angle),
            }

            ok_sz = size_consistency_check(poly, intr, Z_use_m, cfg.box_w_mm, cfg.box_h_mm, rel_err_max=cfg.size_rel_err_max)
            if (not ok_sz) or is_jump(prev_valid, cur, cfg):
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

                last_disp.update(cur)
                if cfg.show_preview:
                    draw_overlay_xyz_angle(vis, cur["Xmm"], cur["Ymm"], cur["Zmm"], cur["angle"], cfg)
                    cv2.imshow(cfg.preview_win_name, vis)
                    if (cv2.waitKey(1) & 0xFF) == 27:
                        return None
                continue

            # accept
            consec_skips = 0
            prev_valid = cur
            accepted_xyza.append([cur["Xmm"], cur["Ymm"], cur["Zmm"], cur["angle"]])

            last_disp.update(cur)

            if cfg.show_preview:
                draw_overlay_xyz_angle(vis, cur["Xmm"], cur["Ymm"], cur["Zmm"], cur["angle"], cfg)
                cv2.imshow(cfg.preview_win_name, vis)
                if (cv2.waitKey(1) & 0xFF) == 27:
                    return None

            if cfg.print_selected_each_accept:
                print(f"[{len(accepted_xyza)}/{cfg.avg_n}] picked=1/{num_boxes}  dist_to_img_center={chosen_dist_px:.1f}px  conf={cf:.2f}  cls={ci}")

            if len(accepted_xyza) >= cfg.avg_n:
                break

        cam_mean = np.mean(np.array(accepted_xyza, dtype=np.float32), axis=0)
        return {
            "cam_x_mm": float(cam_mean[0]),
            "cam_y_mm": float(cam_mean[1]),
            "cam_z_mm": float(cam_mean[2]),
            "angle_deg": float(cam_mean[3]),
        }

    finally:
        try:
            pipeline.stop()
        except Exception:
            pass
        if cfg.show_preview:
            try:
                cv2.destroyWindow(cfg.preview_win_name)
            except Exception:
                pass

# -----------------------------
# Main Test Block
# -----------------------------
if __name__ == "__main__":
    print("Starting measurement...")
    # 기본 설정을 그대로 사용하여 실행
    result = measure_box(DEFAULT_VISION_CONFIG)
    
    if result:
        print("\nMeasurement Result:")
        print(f"  X: {result['cam_x_mm']:.1f} mm")
        print(f"  Y: {result['cam_y_mm']:.1f} mm")
        print(f"  Z: {result['cam_z_mm']:.1f} mm")
        print(f"  Angle: {result['angle_deg']:.2f} deg")
    else:
        print("\nMeasurement failed or timed out.")

Starting measurement...
[1/10] picked=1/2  dist_to_img_center=150.2px  conf=0.96  cls=0
[2/10] picked=1/3  dist_to_img_center=142.9px  conf=0.97  cls=0
[3/10] picked=1/5  dist_to_img_center=130.4px  conf=0.98  cls=0
[4/10] picked=1/4  dist_to_img_center=125.3px  conf=0.96  cls=0
[5/10] picked=1/3  dist_to_img_center=113.1px  conf=0.85  cls=0
[6/10] picked=1/1  dist_to_img_center=50.0px  conf=0.98  cls=0
[7/10] picked=1/1  dist_to_img_center=49.5px  conf=0.98  cls=0
[8/10] picked=1/1  dist_to_img_center=50.1px  conf=0.98  cls=0
[9/10] picked=1/1  dist_to_img_center=50.5px  conf=0.98  cls=0
[10/10] picked=1/1  dist_to_img_center=48.0px  conf=0.99  cls=0

Measurement Result:
  X: -5.8 mm
  Y: 81.8 mm
  Z: 404.6 mm
  Angle: 1.67 deg


In [ ]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple
from dataclasses import dataclass
import time
import datetime
import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.85
    iou_thres: float = 0.75
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # box real size (mm)
    box_w_mm: float = 230.0
    box_h_mm: float = 95.0

    size_rel_err_max: float = 0.25

    # sampling (평균낼 횟수)
    avg_n: int = 10
    
    # depth ROI (meters)
    roi_margin_px: float = 6.0
    min_roi_pixels: int = 120
    mad_thres_m: float = 0.020
    depth_min_m: float = 0.15
    depth_max_m: float = 3.00

    # sanity filters
    z_range_mm: Tuple[float, float] = (150.0, 1200.0)
    
    jump_xy_mm: float = 35.0
    jump_z_mm: float = 60.0
    jump_ang_deg: float = 10.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    preview_win_name: str = "OBB Vision Control"
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2


# 기본 설정 인스턴스
DEFAULT_VISION_CONFIG = VisionConfig(
    # 모델 경로 확인 필수
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)


# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def poly_shrink_towards_center(poly4x2: np.ndarray, margin_px: float):
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    v = p - c
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-6
    return p - (v / norm) * margin_px


def depth_roi_stats(depth_u16: np.ndarray, depth_scale: float, poly4x2: np.ndarray, cfg: VisionConfig):
    h, w = depth_u16.shape[:2]
    poly = np.round(poly4x2).astype(np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.reshape(-1, 1, 2)], 255)

    d = depth_u16[mask == 255].astype(np.float32) * depth_scale
    d = d[(d > 0) & (d >= cfg.depth_min_m) & (d <= cfg.depth_max_m)]
    if d.size == 0:
        return 0.0, 0.0, 0

    med = float(np.median(d))
    mad = float(np.median(np.abs(d - med)))
    return med, mad, int(d.size)


def edges_long_short_px(poly4x2: np.ndarray):
    p = poly4x2.astype(np.float32)
    edges = [np.linalg.norm(p[(i + 1) % 4] - p[i]) for i in range(4)]
    return float(max(edges)), float(min(edges))


def estimate_Z_from_size(poly4x2: np.ndarray, intr, W_mm: float, H_mm: float) -> float:
    long_px, short_px = edges_long_short_px(poly4x2)
    W_m = W_mm / 1000.0
    H_m = H_mm / 1000.0

    if W_m >= H_m:
        Z1 = (intr.fx * W_m) / max(long_px, 1e-6)
        Z2 = (intr.fy * H_m) / max(short_px, 1e-6)
    else:
        Z1 = (intr.fx * H_m) / max(long_px, 1e-6)
        Z2 = (intr.fy * W_m) / max(short_px, 1e-6)

    return float(0.5 * (Z1 + Z2))  # meters


def XY_from_pixel_and_Z(cx: int, cy: int, intr, Z_m: float):
    X = (cx - intr.ppx) / intr.fx * Z_m
    Y = (cy - intr.ppy) / intr.fy * Z_m
    return float(X), float(Y)  # meters


def size_consistency_check(poly4x2, intr, Z_use_m, W_mm, H_mm, rel_err_max=0.25):
    long_px, short_px = edges_long_short_px(poly4x2)

    L1_mm = (long_px * Z_use_m / intr.fx) * 1000.0
    L2_mm = (short_px * Z_use_m / intr.fy) * 1000.0

    W_big = max(W_mm, H_mm)
    H_sml = min(W_mm, H_mm)

    err1 = abs(L1_mm - W_big) / max(1e-6, W_big)
    err2 = abs(L2_mm - H_sml) / max(1e-6, H_sml)

    return (err1 <= rel_err_max) and (err2 <= rel_err_max)


def obb_angle_deg_upright0_rightplus(poly4x2: np.ndarray) -> float:
    p = poly4x2.astype(np.float32)
    c = p.mean(axis=0, keepdims=True)
    q = p - c
    cov = np.cov(q.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    v = eigvecs[:, np.argmax(eigvals)].astype(np.float32)

    vx, vy = float(v[0]), float(v[1])
    if vy < 0:
        vx, vy = -vx, -vy

    angle = float(np.degrees(np.arctan2(vx, vy)))
    angle = -angle
    return angle


def is_jump(prev, cur, cfg: VisionConfig):
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg:
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, angle, cfg: VisionConfig, status_text="", color=(0, 255, 255)):
    line1 = f"cam X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f}  (mm)"
    line2 = f"angle {angle:+.2f} deg"

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    
    # Bottom Status Bar
    if status_text:
        cv2.rectangle(overlay, (6, img.shape[0] - 40), (350, img.shape[0] - 10), (0, 0, 0), -1)

    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    
    if status_text:
        cv2.putText(img, status_text, (10, img.shape[0] - 20), font, fs, color, th, cv2.LINE_AA)


def run_interactive_measurement(cfg: VisionConfig = DEFAULT_VISION_CONFIG):
    print("Loading YOLO model...")
    model = YOLO(cfg.model_path)
    print("Model loaded.")

    print("Initializing RealSense...")
    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = float(depth_sensor.get_depth_scale())

    # Filters
    temporal = rs.temporal_filter()
    spatial = rs.spatial_filter()
    hole = rs.hole_filling_filter()
    spatial.set_option(rs.option.filter_magnitude, 2)
    spatial.set_option(rs.option.filter_smooth_alpha, 0.5)
    spatial.set_option(rs.option.filter_smooth_delta, 20)

    # Window Init
    cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)
    
    # State Variables
    prev_valid = None
    consec_skips = 0
    
    # Data Collection State
    is_collecting = False
    collected_samples = [] # To store the 10 samples
    
    print("\n-------------------------------------------")
    print(" [ Controls ]")
    print(f"  'm' : Trigger Measurement (Collect {cfg.avg_n} samples & Print)")
    print("  's' : Save Screenshot")
    print("  'q' / ESC : Quit")
    print("-------------------------------------------\n")

    try:
        while True:
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            # Process Depth
            depth_frame = spatial.process(depth_frame).as_depth_frame()
            depth_frame = temporal.process(depth_frame).as_depth_frame()
            depth_frame = hole.process(depth_frame).as_depth_frame()

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            depth_u16 = np.asanyarray(depth_frame.get_data())
            
            vis = frame.copy()
            
            # Predict
            results = model.predict(frame, imgsz=cfg.imgsz, conf=cfg.conf_thres, iou=cfg.iou_thres, verbose=False)
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    clss  = obb.cls.cpu().numpy().astype(int)

                    for poly8, cf, ci in zip(polys, confs, clss):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, -float(cf), float(cf), int(ci), poly, cx_det, cy_det))

            # --- Logic to determine current frame status ---
            current_valid_sample = None
            
            if candidates:
                candidates.sort()
                dist2, _ncf, cf, ci, poly, cx_det_f, cy_det_f = candidates[0]
                
                cx = clamp(int(round(cx_det_f)), 0, cfg.width - 1)
                cy = clamp(int(round(cy_det_f)), 0, cfg.height - 1)

                # Draw Visuals (Always)
                poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
                cv2.circle(vis, (cx, cy), 5, (0, 0, 255), -1)

                # Depth Calculation
                poly_shrunk = poly_shrink_towards_center(poly, cfg.roi_margin_px)
                poly_shrunk[:, 0] = np.clip(poly_shrunk[:, 0], 0, cfg.width - 1)
                poly_shrunk[:, 1] = np.clip(poly_shrunk[:, 1], 0, cfg.height - 1)

                Z_roi_m, mad_m, roi_n = depth_roi_stats(depth_u16, depth_scale, poly_shrunk, cfg)
                Z_size_m = estimate_Z_from_size(poly, intr, cfg.box_w_mm, cfg.box_h_mm)

                depth_ok = (Z_roi_m > 0.0 and roi_n >= cfg.min_roi_pixels and mad_m <= cfg.mad_thres_m)
                if depth_ok:
                    alpha = clamp(0.85 - (mad_m / max(1e-6, cfg.mad_thres_m)) * 0.35, 0.55, 0.90)
                    Z_use_m = alpha * Z_roi_m + (1.0 - alpha) * Z_size_m
                else:
                    Z_use_m = Z_size_m

                Z_use_mm = Z_use_m * 1000.0
                
                # Validity Check
                if (cfg.z_range_mm[0] <= Z_use_mm <= cfg.z_range_mm[1]):
                    X_m, Y_m = XY_from_pixel_and_Z(cx, cy, intr, Z_use_m)
                    angle = obb_angle_deg_upright0_rightplus(poly)
                    
                    cur = {
                        "Xmm": X_m * 1000.0,
                        "Ymm": Y_m * 1000.0,
                        "Zmm": Z_use_m * 1000.0,
                        "angle": float(angle),
                    }
                    
                    # Size Check & Jump Filter
                    ok_sz = size_consistency_check(poly, intr, Z_use_m, cfg.box_w_mm, cfg.box_h_mm, rel_err_max=cfg.size_rel_err_max)
                    
                    if ok_sz and not is_jump(prev_valid, cur, cfg):
                        current_valid_sample = cur
                        prev_valid = cur
                        consec_skips = 0
                    else:
                        consec_skips += 1
                        if consec_skips >= cfg.max_consec_skips_reset:
                            prev_valid = None
                            consec_skips = 0
                else:
                    consec_skips += 1
                    if consec_skips >= cfg.max_consec_skips_reset:
                        prev_valid = None
                        consec_skips = 0
            else:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None
                    consec_skips = 0

            # --- Data Collection Logic ---
            if is_collecting:
                status_msg = f"[Collecting: {len(collected_samples)}/{cfg.avg_n}]"
                status_color = (0, 0, 255) # Red while collecting
                
                if current_valid_sample is not None:
                    collected_samples.append([
                        current_valid_sample["Xmm"],
                        current_valid_sample["Ymm"],
                        current_valid_sample["Zmm"],
                        current_valid_sample["angle"]
                    ])
                    # Print progress to terminal (optional)
                    # print(f"Sample {len(collected_samples)}/{cfg.avg_n} captured.")

                # Check if done
                if len(collected_samples) >= cfg.avg_n:
                    arr = np.array(collected_samples, dtype=np.float32)
                    mean_val = np.mean(arr, axis=0)
                    
                    print("\n" + "="*40)
                    print(f" [MEASUREMENT RESULT (Avg of {cfg.avg_n})]")
                    print(f"  X : {mean_val[0]:.1f} mm")
                    print(f"  Y : {mean_val[1]:.1f} mm")
                    print(f"  Z : {mean_val[2]:.1f} mm")
                    print(f"  A : {mean_val[3]:.2f} deg")
                    print("="*40 + "\n")
                    
                    # Reset
                    is_collecting = False
                    collected_samples = []

            else:
                status_msg = "[Monitor Mode] Press 'm' to Measure"
                status_color = (0, 255, 0) # Green while waiting

            # --- Update Display ---
            # Even if not collecting, show the current live value on screen
            if prev_valid is not None:
                draw_overlay_xyz_angle(vis, 
                                       prev_valid["Xmm"], 
                                       prev_valid["Ymm"], 
                                       prev_valid["Zmm"], 
                                       prev_valid["angle"], 
                                       cfg, 
                                       status_msg, 
                                       status_color)
            else:
                 # If nothing valid detected, just show status
                 cv2.putText(vis, status_msg, (10, vis.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)

            # --- Render ---
            cv2.imshow(cfg.preview_win_name, vis)
            
            # --- Key Handling ---
            key = cv2.waitKey(1) & 0xFF
            
            if key == 27 or key == ord('q'): # ESC or q to quit
                break
            
            if key == ord('m'): # Trigger Measurement
                if not is_collecting:
                    print(f"Start collecting {cfg.avg_n} samples...")
                    is_collecting = True
                    collected_samples = []
                else:
                    print("Already collecting... please wait.")

            if key == ord('s'): # Screenshot
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"screenshot_{ts}.png"
                cv2.imwrite(filename, vis)
                print(f"Screenshot saved: {filename}")

    finally:
        try:
            pipeline.stop()
        except Exception:
            pass
        try:
            cv2.destroyAllWindows()
        except Exception:
            pass

# -----------------------------
# Main Entry Point
# -----------------------------
if __name__ == "__main__":
    run_interactive_measurement(DEFAULT_VISION_CONFIG)

Loading YOLO model...
Model loaded.
Initializing RealSense...

-------------------------------------------
 [ Controls ]
  'm' : Trigger Measurement (Collect 10 samples & Print)
  's' : Save Screenshot
  'q' / ESC : Quit
-------------------------------------------

Screenshot saved: screenshot_20260102_130825.png
Screenshot saved: screenshot_20260102_130827.png
Screenshot saved: screenshot_20260102_130831.png
Screenshot saved: screenshot_20260102_130846.png
Screenshot saved: screenshot_20260102_130852.png
Screenshot saved: screenshot_20260102_130855.png
Screenshot saved: screenshot_20260102_130906.png
Screenshot saved: screenshot_20260102_130911.png


In [ ]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple, List
from dataclasses import dataclass
import time
import datetime
import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.85
    iou_thres: float = 0.75
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # box real size (mm) - [중요] 실제 박스 크기를 정확히 입력해야 PnP가 정확합니다.
    box_w_mm: float = 97.0   # 짧은 변 (Width)
    box_h_mm: float = 232.0  # 긴 변 (Height)
    # 3D 모델 정의 시 (0,0,0)을 중심으로 긴 쪽을 Y축, 짧은 쪽을 X축으로 할지 결정해야 합니다.
    # 여기서는 직관적으로 긴 변을 Y(세로), 짧은 변을 X(가로)로 가정하고 코드를 짰습니다.

    size_rel_err_max: float = 0.25

    # sampling (평균낼 횟수)
    avg_n: int = 10
    
    # sanity filters
    jump_xy_mm: float = 50.0  # PnP는 값이 좀 튈 수 있어서 허용범위를 조금 늘림
    jump_z_mm: float = 80.0
    jump_ang_deg: float = 15.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    preview_win_name: str = "OBB Vision Control (SolvePnP)"
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2


# 기본 설정 인스턴스
DEFAULT_VISION_CONFIG = VisionConfig(
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)


# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def order_points(pts):
    """
    4개의 점을 순서대로 정렬합니다: [Top-Left, Top-Right, Bottom-Right, Bottom-Left]
    이 순서가 3D 모델 포인트 순서와 일치해야 solvePnP가 제대로 작동합니다.
    """
    rect = np.zeros((4, 2), dtype="float32")
    
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)] # Top-Left: 합이 가장 작은 점
    rect[2] = pts[np.argmax(s)] # Bottom-Right: 합이 가장 큰 점
    
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)] # Top-Right: 차이가 가장 작은 점 (x가 크고 y가 작음)
    rect[3] = pts[np.argmax(diff)] # Bottom-Left: 차이가 가장 큰 점 (x가 작고 y가 큼)
    
    return rect

def solve_pose_pnp(poly4x2: np.ndarray, intr, cfg: VisionConfig) -> Tuple[float, float, float, float]:
    """
    SolvePnP를 사용하여 3D 위치(X, Y, Z)와 각도를 계산합니다.
    """
    # 1. 2D Image Points (정렬됨)
    image_points = order_points(poly4x2)

    # 2. 3D Object Points (실제 박스 모델 정의)
    # 박스 중심을 (0,0,0)으로 잡고, 실제 mm 단위 좌표를 정의합니다.
    # 순서는 order_points와 똑같이: TL, TR, BR, BL
    # 가로(짧은변)가 X축, 세로(긴변)가 Y축이라고 가정
    w = cfg.box_w_mm
    h = cfg.box_h_mm
    
    # 만약 YOLO가 감지한 박스가 가로로 누워있다면 w, h를 바꿔줘야 할 수도 있습니다.
    # 여기서는 간단히 긴 변을 기준으로 정렬한다고 가정 (Perspective 문제로 인해 약간의 오차는 있을 수 있음)
    # 더 정교하게 하려면 긴 변과 짧은 변의 픽셀 길이를 비교해서 매칭해야 합니다.
    
    # 현재 감지된 픽셀상의 가로/세로 비율 확인
    len_top = np.linalg.norm(image_points[0] - image_points[1])
    len_side = np.linalg.norm(image_points[1] - image_points[2])
    
    if len_top > len_side:
        # 감지된 박스가 가로로 깁니다. 모델도 가로로 길게 설정
        real_w, real_h = max(w, h), min(w, h)
    else:
        # 감지된 박스가 세로로 깁니다. 모델도 세로로 길게 설정
        real_w, real_h = min(w, h), max(w, h)

    sx = real_w / 2.0
    sy = real_h / 2.0

    object_points = np.array([
        [-sx, -sy, 0], # Top-Left
        [ sx, -sy, 0], # Top-Right
        [ sx,  sy, 0], # Bottom-Right
        [-sx,  sy, 0]  # Bottom-Left
    ], dtype=np.float32)

    # 3. Camera Matrix 구성
    camera_matrix = np.array([
        [intr.fx, 0, intr.ppx],
        [0, intr.fy, intr.ppy],
        [0, 0, 1]
    ], dtype=np.float32)
    
    #dist_coeffs = np.zeros((4, 1)) # 왜곡은 없거나 이미 보정되었다고 가정
    dist_coeffs = np.array([-0.03841, 0.09418, -0.01255, 0.00551, -0.04916], dtype=np.float32)

    # 4. Solve PnP
    success, rvec, tvec = cv2.solvePnP(object_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)

    if not success:
        return 0.0, 0.0, 0.0, 0.0

    # 5. 결과 변환
    # tvec은 카메라 좌표계에서의 [X, Y, Z] (mm 단위)
    X_mm = float(tvec[0])
    Y_mm = float(tvec[1])
    Z_mm = float(tvec[2])

    # Rotation Vector -> Euler Angle (Z축 회전만 필요)
    rmat, _ = cv2.Rodrigues(rvec)
    
    # 회전 행렬에서 Yaw (Z축 회전) 추출
    # sy = math.sqrt(rmat[0,0] * rmat[0,0] +  rmat[1,0] * rmat[1,0])
    # 여기서는 간단히 atan2 사용
    angle_rad = np.arctan2(rmat[1, 0], rmat[0, 0])
    angle_deg = np.degrees(angle_rad)

    # 각도 보정 ( -180 ~ 180 ) -> 로봇에 맞게 조정 필요시 수정
    # 일반적으로 박스는 90도 대칭이므로 -45 ~ 45로 맞추거나 그대로 사용
    
    return X_mm, Y_mm, Z_mm, angle_deg


def is_jump(prev, cur, cfg: VisionConfig):
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    # 각도는 PnP에서 180도 뒤집히는 경우가 있어 단순 차이 비교시 주의 (여기선 일단 유지)
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg: 
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, angle, cfg: VisionConfig, status_text="", color=(0, 255, 255)):
    line1 = f"PnP X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f} (mm)"
    line2 = f"Angle {angle:+.2f} deg"

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    
    if status_text:
        cv2.rectangle(overlay, (6, img.shape[0] - 40), (350, img.shape[0] - 10), (0, 0, 0), -1)

    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (0, 255, 255), th, cv2.LINE_AA) # Cyan color for PnP
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    
    if status_text:
        cv2.putText(img, status_text, (10, img.shape[0] - 20), font, fs, color, th, cv2.LINE_AA)


def run_interactive_measurement(cfg: VisionConfig = DEFAULT_VISION_CONFIG):
    print("Loading YOLO model...")
    model = YOLO(cfg.model_path)
    print("Model loaded.")

    print("Initializing RealSense...")
    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    # Window Init
    cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)
    
    # State Variables
    prev_valid = None
    consec_skips = 0
    
    # Data Collection State
    is_collecting = False
    collected_samples = [] 
    
    print("\n-------------------------------------------")
    print(" [ Controls ]")
    print(f"  'm' : Trigger Measurement (Collect {cfg.avg_n} samples)")
    print("  's' : Save Screenshot")
    print("  'q' / ESC : Quit")
    print("-------------------------------------------\n")

    try:
        while True:
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            if not color_frame:
                continue

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            
            vis = frame.copy()
            
            # Predict
            results = model.predict(frame, imgsz=cfg.imgsz, conf=cfg.conf_thres, iou=cfg.iou_thres, verbose=False)
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    
                    for poly8, cf in zip(polys, confs):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        
                        # 중심점 단순 계산 (거리 필터링용)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, float(cf), poly))

            current_valid_sample = None
            
            if candidates:
                # 화면 중앙에 가장 가까운 박스 선택
                candidates.sort(key=lambda x: x[0])
                _, cf, poly = candidates[0]
                
                # --- [핵심 변경] SolvePnP로 좌표 계산 ---
                # 이제 Depth Map을 직접 찍는게 아니라, 2D 좌표와 실제 크기로 역산합니다.
                X_pnp, Y_pnp, Z_pnp, Ang_pnp = solve_pose_pnp(poly, intr, cfg)

                # 그리기
                poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
                
                # 투영된 중심점 그리기 (PnP 결과가 맞는지 시각적 확인)
                # 역으로 3D(0,0,0)을 2D로 투영해봅니다.
                if Z_pnp > 0:
                    # 단순 시각화를 위해 화면상 중심(근사)에 점 찍기
                    cx_vis = int(np.mean(poly[:,0]))
                    cy_vis = int(np.mean(poly[:,1]))
                    cv2.circle(vis, (cx_vis, cy_vis), 5, (0, 0, 255), -1)

                # 값 튐 방지 필터
                cur = {
                    "Xmm": X_pnp,
                    "Ymm": Y_pnp,
                    "Zmm": Z_pnp,
                    "angle": Ang_pnp,
                }
                
                # 0값이 나오거나(실패), 범위가 너무 이상하면 스킵
                if Z_pnp > 100.0: # 최소 10cm 이상일때만 유효
                    if not is_jump(prev_valid, cur, cfg):
                        current_valid_sample = cur
                        prev_valid = cur
                        consec_skips = 0
                    else:
                        consec_skips += 1
                        if consec_skips >= cfg.max_consec_skips_reset:
                            prev_valid = None
                else:
                    consec_skips += 1
            else:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None

            # --- Data Collection Logic ---
            if is_collecting:
                status_msg = f"[Collecting: {len(collected_samples)}/{cfg.avg_n}]"
                status_color = (0, 0, 255)
                
                if current_valid_sample is not None:
                    collected_samples.append([
                        current_valid_sample["Xmm"],
                        current_valid_sample["Ymm"],
                        current_valid_sample["Zmm"],
                        current_valid_sample["angle"]
                    ])

                if len(collected_samples) >= cfg.avg_n:
                    arr = np.array(collected_samples, dtype=np.float32)
                    mean_val = np.mean(arr, axis=0)
                    
                    print("\n" + "="*40)
                    print(f" [SolvePnP RESULT (Avg of {cfg.avg_n})]")
                    print(f"  X : {mean_val[0]:.1f} mm")
                    print(f"  Y : {mean_val[1]:.1f} mm")
                    print(f"  Z : {mean_val[2]:.1f} mm")
                    print(f"  A : {mean_val[3]:.2f} deg")
                    print("="*40 + "\n")
                    
                    is_collecting = False
                    collected_samples = []

            else:
                status_msg = "[Monitor Mode] Press 'm' to Measure"
                status_color = (0, 255, 0)

            # --- Update Display ---
            if prev_valid is not None:
                draw_overlay_xyz_angle(vis, 
                                       prev_valid["Xmm"], 
                                       prev_valid["Ymm"], 
                                       prev_valid["Zmm"], 
                                       prev_valid["angle"], 
                                       cfg, 
                                       status_msg, 
                                       status_color)
            else:
                 cv2.putText(vis, status_msg, (10, vis.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)

            # --- Render ---
            cv2.imshow(cfg.preview_win_name, vis)
            
            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord('q'):
                break
            if key == ord('m'):
                if not is_collecting:
                    print(f"Start collecting {cfg.avg_n} samples...")
                    is_collecting = True
                    collected_samples = []
            if key == ord('s'):
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                cv2.imwrite(f"pnp_screenshot_{ts}.png", vis)
                print("Screenshot saved.")

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    run_interactive_measurement(DEFAULT_VISION_CONFIG)

Loading YOLO model...
Model loaded.
Initializing RealSense...

-------------------------------------------
 [ Controls ]
  'm' : Trigger Measurement (Collect 10 samples)
  's' : Save Screenshot
  'q' / ESC : Quit
-------------------------------------------



/tmp/ipykernel_3215235/2458464983.py:136: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  X_mm = float(tvec[0])
/tmp/ipykernel_3215235/2458464983.py:137: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Y_mm = float(tvec[1])
/tmp/ipykernel_3215235/2458464983.py:138: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z_mm = float(tvec[2])


Screenshot saved.


In [12]:
from __future__ import annotations
from typing import Optional, Dict, Any, Tuple, List
from dataclasses import dataclass
import time
import datetime
import numpy as np
import cv2
import pyrealsense2 as rs
from ultralytics import YOLO

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class VisionConfig:
    # model
    model_path: str
    conf_thres: float = 0.85
    iou_thres: float = 0.75
    imgsz: int = 640

    # camera stream
    width: int = 640
    height: int = 480
    fps: int = 30

    # box real size (mm) - [중요] 실제 박스 크기를 정확히 입력해야 PnP가 정확합니다.
    box_w_mm: float = 97.0   # 짧은 변 (Width)
    box_h_mm: float = 232.0  # 긴 변 (Height)
    # 3D 모델 정의 시 (0,0,0)을 중심으로 긴 쪽을 Y축, 짧은 쪽을 X축으로 할지 결정해야 합니다.
    # 여기서는 직관적으로 긴 변을 Y(세로), 짧은 변을 X(가로)로 가정하고 코드를 짰습니다.

    size_rel_err_max: float = 0.25

    # sampling (평균낼 횟수)
    avg_n: int = 10
    
    # sanity filters
    jump_xy_mm: float = 50.0  # PnP는 값이 좀 튈 수 있어서 허용범위를 조금 늘림
    jump_z_mm: float = 80.0
    jump_ang_deg: float = 15.0

    max_consec_skips_reset: int = 15

    # preview / overlay
    preview_win_name: str = "OBB Vision Control (SolvePnP)"
    overlay_font_scale: float = 0.6
    overlay_thickness: int = 2


# 기본 설정 인스턴스
DEFAULT_VISION_CONFIG = VisionConfig(
    model_path=r"/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/best.pt"
)


# -----------------------------
# Local helpers
# -----------------------------
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def order_points(pts):
    """
    4개의 점을 순서대로 정렬합니다: [Top-Left, Top-Right, Bottom-Right, Bottom-Left]
    이 순서가 3D 모델 포인트 순서와 일치해야 solvePnP가 제대로 작동합니다.
    """
    rect = np.zeros((4, 2), dtype="float32")
    
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)] # Top-Left: 합이 가장 작은 점
    rect[2] = pts[np.argmax(s)] # Bottom-Right: 합이 가장 큰 점
    
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)] # Top-Right: 차이가 가장 작은 점 (x가 크고 y가 작음)
    rect[3] = pts[np.argmax(diff)] # Bottom-Left: 차이가 가장 큰 점 (x가 작고 y가 큼)
    
    return rect

def solve_pose_pnp(poly4x2: np.ndarray, intr, cfg: VisionConfig) -> Tuple[float, float, float, float]:
    """
    SolvePnP를 사용하여 3D 위치(X, Y, Z)와 각도를 계산합니다.
    """
    # 1. 2D Image Points (정렬됨)
    image_points = order_points(poly4x2)

    # 2. 3D Object Points (실제 박스 모델 정의)
    # 박스 중심을 (0,0,0)으로 잡고, 실제 mm 단위 좌표를 정의합니다.
    # 순서는 order_points와 똑같이: TL, TR, BR, BL
    # 가로(짧은변)가 X축, 세로(긴변)가 Y축이라고 가정
    w = cfg.box_w_mm
    h = cfg.box_h_mm
    
    # 만약 YOLO가 감지한 박스가 가로로 누워있다면 w, h를 바꿔줘야 할 수도 있습니다.
    # 여기서는 간단히 긴 변을 기준으로 정렬한다고 가정 (Perspective 문제로 인해 약간의 오차는 있을 수 있음)
    # 더 정교하게 하려면 긴 변과 짧은 변의 픽셀 길이를 비교해서 매칭해야 합니다.
    
    # 현재 감지된 픽셀상의 가로/세로 비율 확인
    len_top = np.linalg.norm(image_points[0] - image_points[1])
    len_side = np.linalg.norm(image_points[1] - image_points[2])
    
    if len_top > len_side:
        # 감지된 박스가 가로로 깁니다. 모델도 가로로 길게 설정
        real_w, real_h = max(w, h), min(w, h)
    else:
        # 감지된 박스가 세로로 깁니다. 모델도 세로로 길게 설정
        real_w, real_h = min(w, h), max(w, h)

    sx = real_w / 2.0
    sy = real_h / 2.0

    object_points = np.array([
        [-sx, -sy, 0], # Top-Left
        [ sx, -sy, 0], # Top-Right
        [ sx,  sy, 0], # Bottom-Right
        [-sx,  sy, 0]  # Bottom-Left
    ], dtype=np.float32)

    # 3. Camera Matrix 구성
    camera_matrix = np.array([
        [intr.fx, 0, intr.ppx],
        [0, intr.fy, intr.ppy],
        [0, 0, 1]
    ], dtype=np.float32)
    
    # ==========================================================
    # [수정됨] 캘리브레이션 결과 적용
    # ==========================================================
    # 측정된 왜곡 계수: k1, k2, p1, p2, k3
    dist_coeffs = np.array([-0.03841, 0.09418, -0.01255, 0.00551, -0.04916], dtype=np.float32)

    # 4. Solve PnP
    success, rvec, tvec = cv2.solvePnP(object_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)

    if not success:
        return 0.0, 0.0, 0.0, 0.0

    # 5. 결과 변환
    # tvec은 카메라 좌표계에서의 [X, Y, Z] (mm 단위)
    X_mm = float(tvec[0])
    Y_mm = float(tvec[1])
    Z_mm = float(tvec[2])

    # Rotation Vector -> Euler Angle (Z축 회전만 필요)
    rmat, _ = cv2.Rodrigues(rvec)
    
    # 회전 행렬에서 Yaw (Z축 회전) 추출
    # sy = math.sqrt(rmat[0,0] * rmat[0,0] +  rmat[1,0] * rmat[1,0])
    # 여기서는 간단히 atan2 사용
    angle_rad = np.arctan2(rmat[1, 0], rmat[0, 0])
    angle_deg = np.degrees(angle_rad)

    # 각도 보정 ( -180 ~ 180 ) -> 로봇에 맞게 조정 필요시 수정
    # 일반적으로 박스는 90도 대칭이므로 -45 ~ 45로 맞추거나 그대로 사용
    
    return X_mm, Y_mm, Z_mm, angle_deg


def is_jump(prev, cur, cfg: VisionConfig):
    if prev is None:
        return False
    if abs(cur["Xmm"] - prev["Xmm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Ymm"] - prev["Ymm"]) > cfg.jump_xy_mm:
        return True
    if abs(cur["Zmm"] - prev["Zmm"]) > cfg.jump_z_mm:
        return True
    # 각도는 PnP에서 180도 뒤집히는 경우가 있어 단순 차이 비교시 주의 (여기선 일단 유지)
    if abs(cur["angle"] - prev["angle"]) > cfg.jump_ang_deg: 
        return True
    return False


def draw_overlay_xyz_angle(img, Xmm, Ymm, Zmm, angle, cfg: VisionConfig, status_text="", color=(0, 255, 255)):
    line1 = f"PnP X {Xmm:+.1f}  Y {Ymm:+.1f}  Z {Zmm:+.1f} (mm)"
    line2 = f"Angle {angle:+.2f} deg"

    x, y = 10, 14
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs = cfg.overlay_font_scale
    th = cfg.overlay_thickness

    (w1, h1), _ = cv2.getTextSize(line1, font, fs, th)
    (w2, h2), _ = cv2.getTextSize(line2, font, fs, th)
    w = max(w1, w2)
    h = h1 + h2 + 18

    overlay = img.copy()
    cv2.rectangle(overlay, (6, 6), (6 + w + 12, 6 + h), (0, 0, 0), -1)
    
    if status_text:
        cv2.rectangle(overlay, (6, img.shape[0] - 40), (350, img.shape[0] - 10), (0, 0, 0), -1)

    cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    cv2.putText(img, line1, (x, y + 18), font, fs, (0, 255, 255), th, cv2.LINE_AA) # Cyan color for PnP
    cv2.putText(img, line2, (x, y + 18 + h1 + 6), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    
    if status_text:
        cv2.putText(img, status_text, (10, img.shape[0] - 20), font, fs, color, th, cv2.LINE_AA)


def run_interactive_measurement(cfg: VisionConfig = DEFAULT_VISION_CONFIG):
    print("Loading YOLO model...")
    model = YOLO(cfg.model_path)
    print("Model loaded.")

    print("Initializing RealSense...")
    pipeline = rs.pipeline()
    rs_cfg = rs.config()
    rs_cfg.enable_stream(rs.stream.color, cfg.width, cfg.height, rs.format.bgr8, cfg.fps)
    rs_cfg.enable_stream(rs.stream.depth, cfg.width, cfg.height, rs.format.z16, cfg.fps)

    profile = pipeline.start(rs_cfg)
    align = rs.align(rs.stream.color)

    # Window Init
    cv2.namedWindow(cfg.preview_win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(cfg.preview_win_name, cfg.width, cfg.height)
    
    # State Variables
    prev_valid = None
    consec_skips = 0
    
    # Data Collection State
    is_collecting = False
    collected_samples = [] 
    
    print("\n-------------------------------------------")
    print(" [ Controls ]")
    print(f"  'm' : Trigger Measurement (Collect {cfg.avg_n} samples)")
    print("  's' : Save Screenshot")
    print("  'q' / ESC : Quit")
    print("-------------------------------------------\n")

    try:
        while True:
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            if not color_frame:
                continue

            frame = np.asanyarray(color_frame.get_data())
            intr = color_frame.profile.as_video_stream_profile().get_intrinsics()
            
            vis = frame.copy()
            
            # Predict
            results = model.predict(frame, imgsz=cfg.imgsz, conf=cfg.conf_thres, iou=cfg.iou_thres, verbose=False)
            r = results[0]

            candidates = []
            img_cx = (cfg.width - 1) * 0.5
            img_cy = (cfg.height - 1) * 0.5

            if getattr(r, "obb", None) is not None and r.obb is not None:
                obb = r.obb
                if obb.xyxyxyxy is not None and len(obb.xyxyxyxy) > 0:
                    polys = obb.xyxyxyxy.cpu().numpy()
                    confs = obb.conf.cpu().numpy().astype(float)
                    
                    for poly8, cf in zip(polys, confs):
                        if float(cf) < cfg.conf_thres:
                            continue
                        poly = poly8.reshape(4, 2)
                        
                        # 중심점 단순 계산 (거리 필터링용)
                        cx_det = float(np.mean(poly[:, 0]))
                        cy_det = float(np.mean(poly[:, 1]))
                        dx = cx_det - img_cx
                        dy = cy_det - img_cy
                        dist2 = dx * dx + dy * dy
                        candidates.append((dist2, float(cf), poly))

            current_valid_sample = None
            
            if candidates:
                # 화면 중앙에 가장 가까운 박스 선택
                candidates.sort(key=lambda x: x[0])
                _, cf, poly = candidates[0]
                
                # --- [핵심 변경] SolvePnP로 좌표 계산 ---
                # 이제 Depth Map을 직접 찍는게 아니라, 2D 좌표와 실제 크기로 역산합니다.
                X_pnp, Y_pnp, Z_pnp, Ang_pnp = solve_pose_pnp(poly, intr, cfg)

                # 그리기
                poly_i = np.round(poly).astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(vis, [poly_i], True, (0, 255, 0), 2)
                
                # 투영된 중심점 그리기 (PnP 결과가 맞는지 시각적 확인)
                # 역으로 3D(0,0,0)을 2D로 투영해봅니다.
                if Z_pnp > 0:
                    # 단순 시각화를 위해 화면상 중심(근사)에 점 찍기
                    cx_vis = int(np.mean(poly[:,0]))
                    cy_vis = int(np.mean(poly[:,1]))
                    cv2.circle(vis, (cx_vis, cy_vis), 5, (0, 0, 255), -1)

                # 값 튐 방지 필터
                cur = {
                    "Xmm": X_pnp,
                    "Ymm": Y_pnp,
                    "Zmm": Z_pnp,
                    "angle": Ang_pnp,
                }
                
                # 0값이 나오거나(실패), 범위가 너무 이상하면 스킵
                if Z_pnp > 100.0: # 최소 10cm 이상일때만 유효
                    if not is_jump(prev_valid, cur, cfg):
                        current_valid_sample = cur
                        prev_valid = cur
                        consec_skips = 0
                    else:
                        consec_skips += 1
                        if consec_skips >= cfg.max_consec_skips_reset:
                            prev_valid = None
                else:
                    consec_skips += 1
            else:
                consec_skips += 1
                if consec_skips >= cfg.max_consec_skips_reset:
                    prev_valid = None

            # --- Data Collection Logic ---
            if is_collecting:
                status_msg = f"[Collecting: {len(collected_samples)}/{cfg.avg_n}]"
                status_color = (0, 0, 255)
                
                if current_valid_sample is not None:
                    collected_samples.append([
                        current_valid_sample["Xmm"],
                        current_valid_sample["Ymm"],
                        current_valid_sample["Zmm"],
                        current_valid_sample["angle"]
                    ])

                if len(collected_samples) >= cfg.avg_n:
                    arr = np.array(collected_samples, dtype=np.float32)
                    mean_val = np.mean(arr, axis=0)
                    
                    print("\n" + "="*40)
                    print(f" [SolvePnP RESULT (Avg of {cfg.avg_n})]")
                    print(f"  X : {mean_val[0]:.1f} mm")
                    print(f"  Y : {mean_val[1]:.1f} mm")
                    print(f"  Z : {mean_val[2]:.1f} mm")
                    print(f"  A : {mean_val[3]:.2f} deg")
                    print("="*40 + "\n")
                    
                    is_collecting = False
                    collected_samples = []

            else:
                status_msg = "[Monitor Mode] Press 'm' to Measure"
                status_color = (0, 255, 0)

            # --- Update Display ---
            if prev_valid is not None:
                draw_overlay_xyz_angle(vis, 
                                       prev_valid["Xmm"], 
                                       prev_valid["Ymm"], 
                                       prev_valid["Zmm"], 
                                       prev_valid["angle"], 
                                       cfg, 
                                       status_msg, 
                                       status_color)
            else:
                 cv2.putText(vis, status_msg, (10, vis.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)

            # --- Render ---
            cv2.imshow(cfg.preview_win_name, vis)
            
            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord('q'):
                break
            if key == ord('m'):
                if not is_collecting:
                    print(f"Start collecting {cfg.avg_n} samples...")
                    is_collecting = True
                    collected_samples = []
            if key == ord('s'):
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                cv2.imwrite(f"pnp_screenshot_{ts}.png", vis)
                print("Screenshot saved.")

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    run_interactive_measurement(DEFAULT_VISION_CONFIG)

Loading YOLO model...
Model loaded.
Initializing RealSense...

-------------------------------------------
 [ Controls ]
  'm' : Trigger Measurement (Collect 10 samples)
  's' : Save Screenshot
  'q' / ESC : Quit
-------------------------------------------



/tmp/ipykernel_3215235/2903532632.py:140: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  X_mm = float(tvec[0])
/tmp/ipykernel_3215235/2903532632.py:141: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Y_mm = float(tvec[1])
/tmp/ipykernel_3215235/2903532632.py:142: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z_mm = float(tvec[2])


Screenshot saved.
Screenshot saved.
Screenshot saved.


### Realsense d405 카메라 seg

### Oak 카메라